# 03 - Entrainement des modeles avec MLflow

**Auteur :** Gregory CRESPIN | **Date :** 30/01/2026 | **Version :** 1.0

---

DESCRIPTIF : Ce notebook entraine plusieurs modeles de classification (Logistic
Regression, Random Forest, XGBoost, LightGBM, MLP). Chaque modele est evalue
avec validation croisee (AUC-PR, precision/recall au seuil optimal) et les
resultats sont enregistres dans MLflow pour comparaison.

In [1]:
# Importation des bibliothèques de base
import pandas as pd  # Pour manipuler les DataFrames
import numpy as np  # Pour les calculs numériques

# MLflow : bibliothèque pour le suivi des expériences de machine learning
# Elle enregistre les paramètres, métriques et modèles pour comparaison
import mlflow  # Bibliothèque principale MLflow
import mlflow.sklearn  # Module MLflow pour enregistrer les modèles scikit-learn

# scikit-learn : bibliothèque principale de machine learning en Python
from sklearn.model_selection import StratifiedKFold, cross_val_score
# StratifiedKFold : validation croisée stratifiée (garde la proportion de classes dans chaque fold)
# cross_val_score : fonction pour calculer les scores de validation croisée
from sklearn.base import clone  # clone : créer une copie d'un estimateur (modèle)
from tqdm.auto import tqdm  # tqdm : bibliothèque pour afficher des barres de progression
from sklearn.ensemble import RandomForestClassifier  # Random Forest : modèle d'ensemble d'arbres
from sklearn.linear_model import LogisticRegression  # Régression logistique : modèle linéaire
from sklearn.neural_network import MLPClassifier  # MLP : Multi-Layer Perceptron (réseau de neurones)
from sklearn.metrics import average_precision_score, accuracy_score, confusion_matrix
# average_precision_score : AUC-PR (métrique adaptée aux classes déséquilibrées)
# accuracy_score : précision globale
# confusion_matrix : matrice de confusion (TN, FP, FN, TP)
from sklearn.preprocessing import StandardScaler  # StandardScaler : normalisation des données

# imbalanced-learn : bibliothèque pour gérer les classes déséquilibrées
from imblearn.over_sampling import SMOTE
# SMOTE : Synthetic Minority Over-sampling Technique (création d'exemples synthétiques de la classe minoritaire)

# PyTorch : bibliothèque de deep learning
import torch  # Bibliothèque principale PyTorch
import torch.nn as nn  # nn : module pour créer des réseaux de neurones
# skorch : wrapper qui permet d'utiliser PyTorch comme scikit-learn
from skorch import NeuralNetClassifier  # NeuralNetClassifier : classificateur PyTorch compatible sklearn
from skorch.callbacks import EarlyStopping  # EarlyStopping : arrêt anticipé si pas d'amélioration

# XGBoost et LightGBM : bibliothèques de gradient boosting (modèles d'ensemble très performants)
import xgboost as xgb  # XGBoost : gradient boosting optimisé
import lightgbm as lgb  # LightGBM : gradient boosting rapide et efficace

# Ignorer les avertissements pour une sortie plus propre
import warnings
warnings.filterwarnings('ignore')

# Ajouter le répertoire parent au chemin Python pour importer nos modules
import sys
sys.path.append('..')
# Importer nos fonctions personnalisées
from src.metrics import evaluate_model, print_metrics  # Fonctions d'évaluation des modèles
from utils.business_cost import find_optimal_threshold  # Fonction pour trouver le seuil optimal

# Configuration MLflow
# Définir où stocker les runs MLflow (dans un dossier local)
mlflow.set_tracking_uri("file:../mlruns")
# Définir le nom de l'expérience (tous les runs seront groupés sous ce nom)
mlflow.set_experiment("credit_scoring")

<Experiment: artifact_location='file:///app/notebooks/../mlruns/459041604418719105', creation_time=1769779108975, experiment_id='459041604418719105', last_update_time=1769779108975, lifecycle_stage='active', name='credit_scoring', tags={'mlflow.experimentKind': 'custom_model_development'}>

## 1. Chargement des données préparées

In [2]:
# Charger les données préparées depuis les fichiers CSV
# X_train : features (variables explicatives) pour l'entraînement
X_train = pd.read_csv('../data/X_train_prepared.csv')
# y_train : variable cible (ce qu'on veut prédire) pour l'entraînement
# .iloc[:, 0] : prendre la première colonne (index 0) du DataFrame
y_train = pd.read_csv('../data/y_train_prepared.csv').iloc[:, 0]

# Afficher des informations sur les données chargées
# .shape retourne (nombre de lignes, nombre de colonnes)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nDistribution des classes:")
# value_counts() : compte le nombre d'occurrences de chaque valeur
# Affiche combien de clients sont dans chaque classe (0 = bon client, 1 = mauvais client)
print(y_train.value_counts())
# Calculer le pourcentage de la classe positive (mauvais clients)
# .mean() : moyenne (si classe = 1, moyenne = proportion de 1)
# * 100 : convertir en pourcentage
print(f"\nPourcentage classe 1: {y_train.mean() * 100:.2f}%")

X_train shape: (307511, 173)
y_train shape: (307511,)

Distribution des classes:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Pourcentage classe 1: 8.07%


## 2. Fonction d'entraînement avec MLflow

In [3]:
def train_model_with_mlflow(model, model_name, X, y, cv_folds=5, sampler=None):
    """
    Entraîne un modèle avec validation croisée et tracking MLflow.
    Affiche une barre de progression pour suivre l'avancement.

    DESCRIPTIF :
    - Si `sampler` est fourni (ex. SMOTE), le reechantillonnage est applique
      uniquement sur les donnees d'entrainement (pas sur la validation) afin
      d'eviter toute fuite de donnees.
    - Pour skorch (PyTorch), convertit automatiquement les DataFrames en arrays numpy
      avec les bons dtypes (float32 pour X, int64 pour y) pour compatibilite PyTorch.
    """
    # Detection si c'est un modele skorch (necessite numpy arrays avec dtype specifique)
    is_skorch = 'NeuralNet' in str(type(model))
    
    # Sauvegarder les noms de colonnes pour reconversion si necessaire
    X_columns = X.columns if isinstance(X, pd.DataFrame) else None
    y_name = y.name if isinstance(y, pd.Series) else None
    
    # Conversion pour skorch si necessaire (des le debut) avec dtype explicite
    if is_skorch:
        # Conversion explicite avec copie pour garantir le dtype
        if isinstance(X, pd.DataFrame):
            X = X.values.astype(np.float32, copy=False)
        elif isinstance(X, np.ndarray):
            X = X.astype(np.float32, copy=False)
        else:
            X = np.array(X, dtype=np.float32)
        
        if isinstance(y, pd.Series):
            y = y.values.astype(np.int64, copy=False)
        elif isinstance(y, np.ndarray):
            y = y.astype(np.int64, copy=False)
        else:
            y = np.array(y, dtype=np.int64)
    
    with mlflow.start_run(run_name=model_name):
        skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        
        # Garder les originaux pour CV
        X_orig = X
        y_orig = y
        
        # Entrainement principal
        print(f"Entrainement: {model_name}...")
        if sampler is not None:
            sampler_main = clone(sampler)
            X_res, y_res = sampler_main.fit_resample(X, y)
            # Pour skorch, s'assurer des dtypes corrects (SMOTE renvoie numpy mais peut etre float64)
            if is_skorch:
                # Conversion explicite avec copie pour garantir le dtype
                X_res = X_res.astype(np.float32, copy=False)
                y_res = y_res.astype(np.int64, copy=False)
            else:
                # Pour d'autres modeles, reconvertir en DataFrame/Series
                if not isinstance(X_res, pd.DataFrame):
                    X_res = pd.DataFrame(X_res, columns=X_columns)
                if not isinstance(y_res, pd.Series):
                    y_res = pd.Series(y_res, name=y_name)
            model.fit(X_res, y_res)
        else:
            model.fit(X, y)

        y_proba = model.predict_proba(X)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)
        
        # Évaluation
        metrics = evaluate_model(y, y_pred, y_proba)
        
        # Validation croisee avec barre de progression
        print(f"Validation croisee ({cv_folds} folds)...")
        cv_scores = []
        # Pour CV, utiliser les donnees originales (deja converties si skorch)
        X_cv = X_orig
        y_cv = y_orig
        
        for train_idx, val_idx in tqdm(skf.split(X_cv, y_cv), total=cv_folds, desc="  CV folds", leave=False):
            model_clone = clone(model)
            if is_skorch:
                X_tr, X_val = X_cv[train_idx], X_cv[val_idx]
                y_tr, y_val = y_cv[train_idx], y_cv[val_idx]
            else:
                X_tr, X_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
                y_tr, y_val = y_cv.iloc[train_idx], y_cv.iloc[val_idx]

            # Reechantillonnage uniquement sur le train du fold
            if sampler is not None:
                sampler_fold = clone(sampler)
                X_tr_res, y_tr_res = sampler_fold.fit_resample(X_tr, y_tr)
                # Pour skorch, s'assurer des dtypes corrects (SMOTE renvoie numpy mais peut etre float64)
                if is_skorch:
                    # Conversion explicite avec copie pour garantir le dtype
                    X_tr_res = X_tr_res.astype(np.float32, copy=False)
                    y_tr_res = y_tr_res.astype(np.int64, copy=False)
                else:
                    # Pour d'autres modeles, reconvertir en DataFrame/Series
                    if not isinstance(X_tr_res, pd.DataFrame):
                        X_tr_res = pd.DataFrame(X_tr_res, columns=X_columns)
                    if not isinstance(y_tr_res, pd.Series):
                        y_tr_res = pd.Series(y_tr_res, name=y_name)
                model_clone.fit(X_tr_res, y_tr_res)
            else:
                model_clone.fit(X_tr, y_tr)

            score = average_precision_score(y_val, model_clone.predict_proba(X_val)[:, 1])
            cv_scores.append(score)
        metrics['cv_auc_pr_mean'] = np.mean(cv_scores)
        metrics['cv_auc_pr_std'] = np.std(cv_scores)
        
        # Logger les métriques dans MLflow
        mlflow.log_metrics({
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1_score': metrics['f1_score'],
            'auc_pr': metrics['auc_pr'],
            'cv_auc_pr_mean': metrics['cv_auc_pr_mean'],
            'cv_auc_pr_std': metrics['cv_auc_pr_std'],
            'business_cost': metrics['business_cost'],
            'business_score': metrics['business_score'],
            'optimal_threshold': metrics['optimal_threshold'],
            'precision_at_optimal': metrics['precision_at_optimal'],
            'recall_at_optimal': metrics['recall_at_optimal']
        })
        
        # Logger les paramètres du modèle
        if hasattr(model, 'get_params'):
            mlflow.log_params(model.get_params())
        
        # Enregistrer le modèle
        mlflow.sklearn.log_model(model, "model")
        
        # Afficher les métriques
        print(f"\n{'='*60}")
        print(f"Modèle: {model_name}")
        print_metrics(metrics)
        
        return model, metrics

## 3. Test de différents modèles

In [4]:
# Preparation des donnees normalisees (utilisees par Logistic Regression et MLP)
# La normalisation est importante pour ces modèles car ils sont sensibles à l'échelle des variables

# StandardScaler pour la régression logistique
# StandardScaler : centre les données (moyenne=0) et réduit (écart-type=1)
lr_scaler = StandardScaler()
# fit_transform() : apprend les paramètres (moyenne, écart-type) ET transforme les données
# pd.DataFrame() : recréer un DataFrame avec les mêmes colonnes et index
X_train_scaled_lr = pd.DataFrame(
    lr_scaler.fit_transform(X_train),  # Appliquer la normalisation
    columns=X_train.columns,  # Garder les mêmes noms de colonnes
    index=X_train.index  # Garder les mêmes index (identifiants)
)

# StandardScaler pour le MLP (réseau de neurones)
# On crée un scaler séparé pour pouvoir réutiliser les paramètres plus tard
scaler_mlp = StandardScaler()
X_train_scaled_mlp = pd.DataFrame(
    scaler_mlp.fit_transform(X_train),  # Appliquer la normalisation
    columns=X_train.columns,  # Garder les mêmes noms de colonnes
    index=X_train.index  # Garder les mêmes index
)

# Reechantillonnage (utile avec ~8% de classe positive)
# SMOTE : crée des exemples synthétiques de la classe minoritaire pour équilibrer les classes
# sampling_strategy=0.2 : augmenter la classe minoritaire à 20% du total (au lieu de 8%)
# random_state=42 : fixer la graine aléatoire pour reproductibilité
# n_jobs=-1 : utiliser tous les cœurs CPU disponibles pour accélérer le traitement
sampler = SMOTE(sampling_strategy=0.2, random_state=42, n_jobs=-1)

# Modele Skorch (PyTorch via interface sklearn)
class SkorchMLPModule(nn.Module):
    def __init__(self, n_in, n_hidden1=128, n_hidden2=64, n_out=2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_in, n_hidden1),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(n_hidden2, n_out),
        )
    def forward(self, x):
        return self.layers(x)

n_features = X_train_scaled_mlp.shape[1]
# Reproducibilite pour PyTorch (skorch n'accepte pas random_state directement)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

skorch_net = NeuralNetClassifier(
    SkorchMLPModule,
    module__n_in=n_features,
    max_epochs=200,
    lr=0.001,
    batch_size=1024,
    optimizer=torch.optim.Adam,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    iterator_train__shuffle=True,
    callbacks=[('early_stop', EarlyStopping(patience=10))],
)

# Dictionnaire pour stocker les resultats
trained_models = {}
all_metrics = {}

## 3a. Modèle 1 : Logistic Regression

In [5]:
lr_model = LogisticRegression(
    class_weight='balanced', max_iter=2000, solver='lbfgs',
    random_state=42, n_jobs=-1
)

lr_trained, lr_metrics = train_model_with_mlflow(
    lr_model, "LogisticRegression_balanced", X_train_scaled_lr, y_train, sampler=sampler
)
trained_models["LogisticRegression_balanced"] = lr_trained
all_metrics["LogisticRegression_balanced"] = lr_metrics

Entrainement: LogisticRegression_balanced...
Validation croisee (5 folds)...


  CV folds:   0%|          | 0/5 [00:00<?, ?it/s]


Modèle: LogisticRegression_balanced
METRIQUES D'EVALUATION

[METRIQUES CLASSIQUES]
  Accuracy:  0.7022
  Precision: 0.1702
  Recall:    0.6937
  F1-Score:  0.2733
  AUC-PR:    0.2460

[MATRICE DE CONFUSION]
  Vrais Negatifs (TN): 198706
  Faux Positifs (FP):  83980
  Faux Negatifs (FN):  7603
  Vrais Positifs (TP): 17222

[METRIQUES METIER]
  Cout metier:        160010.00
  Cout metier min:    159289.00
  Seuil optimal:     0.520
  Score metier:      -159289.00
  Precision (seuil optimal): 0.1771
  Recall (seuil optimal):   0.6696


## 3b. Modèle 2 : Random Forest

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, class_weight='balanced',
    random_state=42, n_jobs=-1
)

rf_trained, rf_metrics = train_model_with_mlflow(
    rf_model, "RandomForest_balanced", X_train, y_train, sampler=sampler
)
trained_models["RandomForest_balanced"] = rf_trained
all_metrics["RandomForest_balanced"] = rf_metrics

Entrainement: RandomForest_balanced...
Validation croisee (5 folds)...


  CV folds:   0%|          | 0/5 [00:00<?, ?it/s]


Modèle: RandomForest_balanced
METRIQUES D'EVALUATION

[METRIQUES CLASSIQUES]
  Accuracy:  0.8158
  Precision: 0.2314
  Recall:    0.5518
  F1-Score:  0.3260
  AUC-PR:    0.2528

[MATRICE DE CONFUSION]
  Vrais Negatifs (TN): 237174
  Faux Positifs (FP):  45512
  Faux Negatifs (FN):  11126
  Vrais Positifs (TP): 13699

[METRIQUES METIER]
  Cout metier:        156772.00
  Cout metier min:    149674.00
  Seuil optimal:     0.450
  Score metier:      -149674.00
  Precision (seuil optimal): 0.1964
  Recall (seuil optimal):   0.6719


## 3c. Modèle 3 : XGBoost

In [7]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42, n_jobs=-1, eval_metric='aucpr'
)

xgb_trained, xgb_metrics = train_model_with_mlflow(
    xgb_model, "XGBoost_balanced", X_train, y_train, sampler=sampler
)
trained_models["XGBoost_balanced"] = xgb_trained
all_metrics["XGBoost_balanced"] = xgb_metrics

Entrainement: XGBoost_balanced...
Validation croisee (5 folds)...


  CV folds:   0%|          | 0/5 [00:00<?, ?it/s]


Modèle: XGBoost_balanced
METRIQUES D'EVALUATION

[METRIQUES CLASSIQUES]
  Accuracy:  0.7084
  Precision: 0.1847
  Recall:    0.7653
  F1-Score:  0.2976
  AUC-PR:    0.3122

[MATRICE DE CONFUSION]
  Vrais Negatifs (TN): 198835
  Faux Positifs (FP):  83851
  Faux Negatifs (FN):  5827
  Vrais Positifs (TP): 18998

[METRIQUES METIER]
  Cout metier:        142121.00
  Cout metier min:    141455.00
  Seuil optimal:     0.530
  Score metier:      -141455.00
  Precision (seuil optimal): 0.1963
  Recall (seuil optimal):   0.7283


## 3d. Modèle 4 : LightGBM

In [8]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
) 

lgb_trained, lgb_metrics = train_model_with_mlflow(
    lgb_model, "LightGBM_balanced", X_train, y_train, sampler=sampler
)
trained_models["LightGBM_balanced"] = lgb_trained
all_metrics["LightGBM_balanced"] = lgb_metrics

Entrainement: LightGBM_balanced...
Validation croisee (5 folds)...


  CV folds:   0%|          | 0/5 [00:00<?, ?it/s]


Modèle: LightGBM_balanced
METRIQUES D'EVALUATION

[METRIQUES CLASSIQUES]
  Accuracy:  0.8576
  Precision: 0.2750
  Recall:    0.4669
  F1-Score:  0.3462
  AUC-PR:    0.2950

[MATRICE DE CONFUSION]
  Vrais Negatifs (TN): 252136
  Faux Positifs (FP):  30550
  Faux Negatifs (FN):  13235
  Vrais Positifs (TP): 11590

[METRIQUES METIER]
  Cout metier:        162900.00
  Cout metier min:    147587.00
  Seuil optimal:     0.350
  Score metier:      -147587.00
  Precision (seuil optimal): 0.1947
  Recall (seuil optimal):   0.6916


## 3e. Modèle 5 : MLPClassifier (scikit-learn)

In [9]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
    alpha=0.001, max_iter=200, random_state=42,
    early_stopping=True, validation_fraction=0.1
)

mlp_trained, mlp_metrics = train_model_with_mlflow(
    mlp_model, "MLPClassifier", X_train_scaled_mlp, y_train, sampler=sampler
)
trained_models["MLPClassifier"] = mlp_trained
all_metrics["MLPClassifier"] = mlp_metrics

Entrainement: MLPClassifier...
Validation croisee (5 folds)...


  CV folds:   0%|          | 0/5 [00:00<?, ?it/s]


Modèle: MLPClassifier
METRIQUES D'EVALUATION

[METRIQUES CLASSIQUES]
  Accuracy:  0.9240
  Precision: 0.5493
  Recall:    0.3261
  F1-Score:  0.4092
  AUC-PR:    0.4417

[MATRICE DE CONFUSION]
  Vrais Negatifs (TN): 276045
  Faux Positifs (FP):  6641
  Faux Negatifs (FN):  16730
  Vrais Positifs (TP): 8095

[METRIQUES METIER]
  Cout metier:        173941.00
  Cout metier min:    111592.00
  Seuil optimal:     0.120
  Score metier:      -111592.00
  Precision (seuil optimal): 0.2659
  Recall (seuil optimal):   0.7604


## 3f. Modèle 6 : SkorchMLP (PyTorch)

In [10]:
# Pour skorch, convertir les donnees en numpy arrays avec les bons dtypes (float32 pour X, int64 pour y)
X_train_scaled_mlp_numpy = X_train_scaled_mlp.values.astype(np.float32, copy=False)
y_train_numpy = y_train.values.astype(np.int64, copy=False)

skorch_trained, skorch_metrics = train_model_with_mlflow(
    skorch_net, "SkorchMLP", X_train_scaled_mlp_numpy, y_train_numpy, sampler=sampler
)
trained_models["SkorchMLP"] = skorch_trained
all_metrics["SkorchMLP"] = skorch_metrics

Entrainement: SkorchMLP...
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1           nan       0.8333       -4.6147  2.7482
      2       -5.5055       0.8333       -6.2462  2.8718
      3       -6.7619       0.8333       -7.2284  2.6548
      4       -7.5887       0.8333       -7.9300  2.5971
      5       -8.2080       0.8333       -8.4799  2.7045
      6       -8.7080       0.8333       -8.9364  2.6639
      7       -9.1319       0.8333       -9.3303  2.6148
      8       -9.5021       0.8333       -9.6794  2.6699
      9       -9.8340       0.8333       -9.9952  2.8023
     10      -10.1365       0.8333      -10.2853  2.9305
     11      -10.4162       0.8333      -10.5550  3.1508
     12      -10.6772       0.8333      -10.8083  2.7231
     13      -10.9239       0.8333      -11.0479  2.6382
     14      -11.1578       0.8333      -11.2762  2.8017
     15      -11.3810       0.8333      -11.4948  2.7387
    

  CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1           nan       0.1667           nan  2.2542
      2           nan       0.1667           nan  2.5159
      3           nan       0.1667           nan  2.3152
      4           nan       0.1667           nan  2.2365
      5           nan       0.1667           nan  2.4108
      6           nan       0.1667           nan  2.2354
      7           nan       0.1667           nan  2.3267
      8           nan       0.1667           nan  2.1692
      9           nan       0.1667           nan  2.5129
Stopping since valid_loss has not improved in the last 10 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1           nan       0.8333       -3.5243  2.4509
      2       -4.3978       0.8333       -5.1276  2.2846
      3       -5.6377       0.8333       -6.1047  2.3118
      4       -6.4652 

2026/02/07 09:28:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmphvn0lhep/model/model.pkl, flavor: sklearn), fall back to return ['scikit-learn==1.3.2', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback.



Modèle: SkorchMLP
METRIQUES D'EVALUATION

[METRIQUES CLASSIQUES]
  Accuracy:  0.0807
  Precision: 0.0807
  Recall:    1.0000
  F1-Score:  0.1494
  AUC-PR:    0.0990

[MATRICE DE CONFUSION]
  Vrais Negatifs (TN): 0
  Faux Positifs (FP):  282686
  Faux Negatifs (FN):  0
  Vrais Positifs (TP): 24825

[METRIQUES METIER]
  Cout metier:        282686.00
  Cout metier min:    282686.00
  Seuil optimal:     0.100
  Score metier:      -282686.00
  Precision (seuil optimal): 0.0807
  Recall (seuil optimal):   1.0000


## 4. Comparaison des modèles

In [11]:
# Comparer les resultats (all_metrics rempli par la boucle d'entrainement)
results = pd.DataFrame(all_metrics).T

print("Comparaison des modeles:")
print(results[['auc_pr', 'cv_auc_pr_mean', 'precision_at_optimal', 'recall_at_optimal', 'business_score', 'optimal_threshold']])

# Identifier le meilleur modele selon le score metier
best_model_name = results['business_score'].idxmax()
print(f"\n[MEILLEUR MODELE] Score metier: {best_model_name}")

Comparaison des modeles:
                               auc_pr  cv_auc_pr_mean  precision_at_optimal  \
LogisticRegression_balanced  0.246047        0.243743              0.177054   
RandomForest_balanced        0.252840        0.197682              0.196444   
XGBoost_balanced             0.312186        0.256609              0.196328   
LightGBM_balanced            0.294953        0.258146              0.194682   
MLPClassifier                0.441709        0.164199              0.265914   
SkorchMLP                    0.098956        0.083771              0.080729   

                             recall_at_optimal  business_score  \
LogisticRegression_balanced           0.669567       -159289.0   
RandomForest_balanced                 0.671944       -149674.0   
XGBoost_balanced                      0.728338       -141455.0   
LightGBM_balanced                     0.691561       -147587.0   
MLPClassifier                         0.760403       -111592.0   
SkorchMLP                

## 5. Lancer l'interface MLflow UI

Pour visualiser les résultats dans l'interface MLflow, exécutez dans un terminal :
```bash
mlflow ui --backend-store-uri file:../mlruns
```
Puis ouvrez http://localhost:5000 dans votre navigateur.